<a href="https://colab.research.google.com/github/ChanchalSaha48/nlp-learning-journey/blob/main/03_machine_learning/03_Regularization%20in%20Logistic%20Regression/regularization_in_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import opendatasets as od
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [4]:
od.download('https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews')

Skipping, found downloaded files in "./imdb-dataset-of-50k-movie-reviews" (use force=True to force download)


In [5]:
df=pd.read_csv('/content/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
from bs4 import BeautifulSoup

def preprocessing(text):
  return (BeautifulSoup(text,'html.parser').get_text()).lower()

In [7]:
# Check preprocessing block

text="hello <br>borun"
preprocessing(text)

'hello borun'

In [8]:
df['review']=df['review'].apply(preprocessing)

In [9]:
df['label']=df['sentiment'].map({
    'negative':0,
    'positive':1
})

## C = 1 / Regularization_Strength

In [10]:
C_values=[0.01,0.1,1,10,100]

In [11]:
comparision_table=[]
for C in C_values:
  model=LogisticRegression(C=C,max_iter=1000)

  tfidf=TfidfVectorizer(max_features=20000)
  x_train,x_test,y_train,y_test=train_test_split(df['review'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])

  # Transform text data using TF-IDF
  x_train_tfidf = tfidf.fit_transform(x_train)
  x_test_tfidf = tfidf.transform(x_test)

  model.fit(x_train_tfidf,y_train)

  y_pred=model.predict(x_test_tfidf)

  y_pred_train=model.predict(x_train_tfidf)

  train_acc=accuracy_score(y_pred_train,y_train)
  train_pre=precision_score(y_pred_train,y_train)
  train_rec=recall_score(y_pred_train,y_train)
  train_f1=f1_score(y_pred_train,y_train)

  test_acc=accuracy_score(y_test,y_pred)
  test_pre=precision_score(y_test,y_pred)
  test_rec=recall_score(y_test,y_pred)
  test_f1=f1_score(y_test,y_pred)

  comparision_table.append([C,train_acc,test_acc,train_pre,test_pre,train_rec,test_rec,train_f1,test_f1])

In [12]:
d=pd.DataFrame(comparision_table,columns=['C','train_acc','test_acc','train_pre','test_pre','train_rec','test_rec','train_f1','test_f1'])

In [13]:

d

,C,train_acc,test_acc,train_pre,test_pre,train_rec,test_rec,train_f1,test_f1
0,0.01,0.814475,0.8117,0.84900,0.792018,0.794163,0.8454,0.820666,0.817839
1,0.10,0.877450,0.8720,0.89575,0.859073,0.864123,0.8900,0.879652,0.874263
2,1.00,0.925925,0.9002,0.93365,0.895298,0.919445,0.9064,0.926493,0.900815
3,10.00,0.968625,0.9005,0.97050,0.897558,0.966874,0.9042,0.968684,0.900867
4,100.00,0.995100,0.8878,0.99525,0.885181,0.994952,0.8912,0.995101,0.888180


### Regularization Experiment

I experimented with different values of C in Logistic Regression.

As C increases, regularization becomes weaker.This allws the model to fit the training data more closely.

However, When C becomes too large, the gap between training and test performance increases, indicating overfitting.

In this experiment, C=1 provided a good balance between training and test performance, while C=100 showed clear signs of overfitting.